# Conversational App for Itinerary Planning

In [1]:
import os
from dotenv import load_dotenv
#from langchain import HuggingFaceHub

from langchain import PromptTemplate, LLMChain

load_dotenv()

#os.environ["HUGGINGFACEHUB_API_TOKEN"]
#retrieving api key
key=os.environ['OPENAI_API_KEY']

In [2]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage
)
from langchain.chains import LLMChain, ConversationChain
from langchain_openai import ChatOpenAI

chat = ChatOpenAI()

## Sample bot with no memory

In [3]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage
)
from langchain.chains import LLMChain, ConversationChain
from langchain_openai import ChatOpenAI

chat = ChatOpenAI()
messages = [
    SystemMessage(content="You are a helpful assistant that help the user to plan an optimized itinerary."),
    HumanMessage(content="I'm going to Rome for 2 days, what can I visit?")
]

# Use invoke instead of the deprecated __call__ method
output = chat.invoke(messages)
print(output.content)

That's great! Rome is a beautiful city with so much to see and do. Here is a suggested itinerary for your 2 days in Rome:

Day 1:
1. Start your day at the iconic Colosseum, an ancient Roman amphitheater that is a must-visit in Rome.
2. From there, visit the Roman Forum, an archaeological site that was once the center of ancient Rome.
3. Head to the nearby Palatine Hill, one of the Seven Hills of Rome and a great place to enjoy panoramic views of the city.
4. After lunch, visit the Pantheon, a magnificent ancient temple turned church known for its impressive dome.
5. Walk to Piazza Navona, a beautiful square lined with cafes, fountains, and baroque architecture.
6. End your day at the Trevi Fountain, where you can toss a coin over your shoulder for good luck.

Day 2:
1. Start your day at Vatican City, the smallest independent state in the world and home to St. Peter's Basilica and the Vatican Museums.
2. Explore St. Peter's Basilica, the largest church in the world and a masterpiece of 

## Adding Memory

In [4]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from typing import List

# Initialize the chat model
chat = ChatOpenAI()

# Create a message history
message_history = ChatMessageHistory()

# Add system message
system_message = SystemMessage(content="You are a helpful assistant.")
message_history.add_message(system_message)

# Create a messages list
messages = message_history.messages + [
    HumanMessage(content="Hi there!")
]

# Run the conversation
response = chat.invoke(messages)

# Add the response to history
message_history.add_message(response)

In [5]:
# Add a new human message for the follow-up question
new_question = HumanMessage(content="What is the most iconic place in Rome?")
message_history.add_message(new_question)

# Get all messages from the history
messages = message_history.messages

# Invoke the chat model with all messages
response = chat.invoke(messages)

# Add the response to history
message_history.add_message(response)

# Print the response
print(response.content)

One of the most iconic places in Rome is the Colosseum. It is an ancient amphitheater that was built in 70-80 AD and is known for its history as a site of gladiatorial contests and other public spectacles. It is a symbol of Roman engineering and architecture, as well as a popular tourist attraction today.


In [6]:
# Add a new human message for the follow-up question
new_question = HumanMessage(content="What kind of other events?")
message_history.add_message(new_question)

# Get all messages from the history
messages = message_history.messages

# Invoke the chat model with all messages
response = chat.invoke(messages)

# Add the response to history
message_history.add_message(response)

# Print the response
print(response.content)

In addition to gladiatorial contests, the Colosseum was also used for other events such as animal hunts, mock sea battles, and executions. It could hold up to 80,000 spectators and was a focal point of Roman entertainment and public gatherings. The Colosseum has a rich history and remains a symbol of Rome's ancient past.


In [7]:
# Get all messages from the message history
messages = message_history.messages

In [8]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_community.chat_message_histories import ChatMessageHistory  # Try community package
from langchain_core.runnables import RunnableSequence
from langchain_openai import ChatOpenAI

# Initialize the chat model if not already done
chat = ChatOpenAI()

# Keep your prompt setup the same
prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        "You are a helpful assistant that help the user to plan an optimized itinerary."
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    HumanMessagePromptTemplate.from_template("{question}")
])

# Set up message history instead of memory
chat_history = ChatMessageHistory()

# Create the chain using the pipe syntax
conversation = prompt | chat

# To use it with chat history, create an invoke function
async def invoke_conversation(question: str):
    # Get messages from chat history
    messages = chat_history.messages
    
    # Invoke conversation
    response = await conversation.ainvoke({
        "chat_history": messages,
        "question": question
    })
    
    # Add the new messages to history
    chat_history.add_user_message(question)
    chat_history.add_ai_message(response.content)
    
    return response

# Non-async version if needed
def invoke_conversation_sync(question: str):
    # Get messages from chat history
    messages = chat_history.messages
    
    # Invoke conversation
    response = conversation.invoke({
        "chat_history": messages,
        "question": question
    })
    
    # Add the new messages to history
    chat_history.add_user_message(question)
    chat_history.add_ai_message(response.content)
    
    return response

In [9]:
while True:
    query = input('you: ')
    if query == 'q':
        break
    
    # Load memory contents
    memory_contents = memory.load_memory_variables({})
    
    # Use invoke() method instead of calling directly
    output = conversation.invoke({
        "chat_history": memory_contents["chat_history"],
        "question": query
    })
    
    # The output structure is different in the new format
    print('User: ', query)
    print('AI system: ', output.content)  # Use .content instead of ['text']
    
    # Save the conversation to memory
    memory.save_context({"question": query}, {"output": output.content})

you:  q


## Adding non parametric knowledge

In [10]:
from langchain.llms import OpenAI
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.document_loaders import PyPDFLoader

import os

from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"]

text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1500,
            chunk_overlap=200
        )

raw_documents = PyPDFLoader('italy_travel.pdf').load()
documents = text_splitter.split_documents(raw_documents)
db = FAISS.from_documents(documents, OpenAIEmbeddings())

In [13]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Create a message history to store the conversation
message_history = ChatMessageHistory()

# Initialize the LLM
llm = ChatOpenAI()

# Create prompt template
prompt = ChatPromptTemplate.from_template("""
Answer the following question based on the provided context:

Context: {context}

Question: {input}
""")

# Create the chain
document_chain = create_stuff_documents_chain(llm, prompt)
retrieval_chain = create_retrieval_chain(db.as_retriever(), document_chain)

# Use invoke instead of run
response = retrieval_chain.invoke({"input": "Give me some review about the Pantheon"})

# Add the question and answer to the message history
message_history.add_user_message("Give me some review about the Pantheon")
message_history.add_ai_message(response["answer"])

# Print the answer
print(response["answer"])

Miskita describes the Pantheon as having an "angelic and non-human design," highlighting the gigantic dome, upper eye, sheer size of the building, and overall harmony. Almudena mentions that it is one of the best-preserved ancient Roman monuments and was the first classical building to be transformed into a church. Overall, visitors seem to be impressed by the historical significance and architectural beauty of the Pantheon.


In [16]:
from langchain_core.prompts import PromptTemplate
from langchain.chains import ConversationalRetrievalChain
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain.memory import ConversationBufferMemory

# Custom template for standalone question generation
custom_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question. 
If you cannot find the answer in the document provided, ignore the document and answer anyway.
Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""

CUSTOM_QUESTION_PROMPT = PromptTemplate.from_template(custom_template)

# Create a chat message history
chat_history = ChatMessageHistory()

# Initialize the memory with the updated approach
memory = ConversationBufferMemory(
    memory_key='chat_history',
    return_messages=True,
    chat_memory=chat_history  # Use the chat_memory parameter
)

# Create the chain
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=db.as_retriever(),
    condense_question_prompt=CUSTOM_QUESTION_PROMPT,
    memory=memory,
    verbose=True
)

# Use invoke instead of run
response = qa_chain.invoke({"question": "What can I visit in India?"})

print(response)



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
What to see
 in Italy
Page 6
Canals of Venice 
la peñita:
 To visit Venice in January is to discover another city different from that found on other occasions in July or August.
Tourists (from myself included) inadvertently "spoil" the best views of the sites. If in summer the city is abuzz with activity, in winter
it has a romantic touch that makes me like it even more, if possible. Getting lost in the alleys away from the sea of humanity,
finding a niche with an carved image, a passageway under a building, or a hidden courtyard awaken in my mind an adventurous
sensations which makes me like Venice most each time I visit. Also, you can get off the beaten track and mix it up with the
inhabitants of t

In [18]:
from langchain.agents.agent_toolkits import create_retriever_tool

tool = create_retriever_tool(
    db.as_retriever(), 
    "italy_travel",
    "Searches and returns documents regarding Italy."
)
tools = [tool]

memory = ConversationBufferMemory(
            memory_key='chat_history',
            return_messages=True
        )

from langchain.agents.agent_toolkits import create_conversational_retrieval_agent

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(temperature = 0)

agent_executor = create_conversational_retrieval_agent(llm, tools, memory_key='chat_history', verbose=True)

In [19]:
agent_executor.invoke({"input": "hi, i'm Vale"})



> Entering new AgentExecutor chain...
Hello Vale! How can I assist you today?

> Finished chain.


{'input': "hi, i'm Vale",
 'chat_history': [HumanMessage(content="hi, i'm Vale", additional_kwargs={}, response_metadata={}),
  AIMessage(content='Hello Vale! How can I assist you today?', additional_kwargs={}, response_metadata={})],
 'output': 'Hello Vale! How can I assist you today?',
 'intermediate_steps': []}

In [20]:
agent_executor.invoke({"input": "Tell me something about Pantheon"})



> Entering new AgentExecutor chain...

Invoking: `italy_travel` with `{'query': 'Pantheon'}`


cafes in the square. The most famous are the Quadri and
Florian. 
Piazza San Marco, 
Venice
4
Historical Monuments
Pantheon
 
Miskita:
 
"Angelic and non-human design," was how
Michelangelo described the Pantheon 14 centuries after its
construction. The highlights are the gigantic dome, the upper
eye, the sheer size of the place, and the harmony of the
whole building. We visited with a Roman guide which is
exactly how you should (or shouldn't) visit the city, especially
since they talk too much and have lots and lots of history of
each building. 
And so we learned things like how the interior
dome was filled with sand during construction. Or not,
because it really is impossible to be sure of anything when
one doesn't know Italian and the guide has a very thick
Roman accent. But we loved the place, especially the
entrance portico with its stout columns. 
 
Almudena:
 
The Pantheon is one of 

{'input': 'Tell me something about Pantheon',
 'chat_history': [HumanMessage(content="hi, i'm Vale", additional_kwargs={}, response_metadata={}),
  AIMessage(content='Hello Vale! How can I assist you today?', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Tell me something about Pantheon', additional_kwargs={}, response_metadata={}),
  AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"query":"Pantheon"}', 'name': 'italy_travel'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 110, 'total_tokens': 128, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-BGnT3xe8vTSzOFvQI5NGMBiovyZ5o', 'finish_reason': 'function_call', 'logprobs': None}, id='run-dc95dc62-01c1-4c5e-

In [21]:
output = agent_executor({"input": "what can I visit in India in 3 days?"})
output['output']



> Entering new AgentExecutor chain...


/var/folders/5s/f9f7n_tj38dfzkc2lkzw67q80000gn/T/ipykernel_66614/180987740.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  output = agent_executor({"input": "what can I visit in India in 3 days?"})


In India, with only 3 days to explore, you can consider visiting some of the following popular destinations:

1. Delhi: Explore the historical landmarks such as Red Fort, India Gate, and Qutub Minar. Visit the bustling markets like Chandni Chowk and enjoy local street food.

2. Agra: Visit the iconic Taj Mahal, a UNESCO World Heritage Site, and explore the Agra Fort. These are must-visit attractions in Agra.

3. Jaipur: Known as the Pink City, Jaipur offers attractions like the Amber Fort, City Palace, Hawa Mahal, and Jantar Mantar. Explore the vibrant culture and heritage of Rajasthan.

These destinations are part of the famous Golden Triangle circuit in India and can give you a glimpse of the rich history, culture, and architecture of the country within a short span of time.

> Finished chain.


'In India, with only 3 days to explore, you can consider visiting some of the following popular destinations:\n\n1. Delhi: Explore the historical landmarks such as Red Fort, India Gate, and Qutub Minar. Visit the bustling markets like Chandni Chowk and enjoy local street food.\n\n2. Agra: Visit the iconic Taj Mahal, a UNESCO World Heritage Site, and explore the Agra Fort. These are must-visit attractions in Agra.\n\n3. Jaipur: Known as the Pink City, Jaipur offers attractions like the Amber Fort, City Palace, Hawa Mahal, and Jantar Mantar. Explore the vibrant culture and heritage of Rajasthan.\n\nThese destinations are part of the famous Golden Triangle circuit in India and can give you a glimpse of the rich history, culture, and architecture of the country within a short span of time.'

## Adding external tools

In [22]:
from langchain import SerpAPIWrapper
from langchain.agents import AgentType, initialize_agent
from langchain.llms import OpenAI
from langchain.tools import BaseTool, StructuredTool, Tool, tool

import os
from dotenv import load_dotenv

load_dotenv()

key = os.environ["SERPAPI_API_KEY"]

search = SerpAPIWrapper()

In [ ]:
tools = [
    Tool.from_function(
        func=search.run,
        name="Search",
        description="useful for when you need to answer questions about current events"
    ),
    create_retriever_tool(
        db.as_retriever(), 
        "italy_travel",
        "Searches and returns documents regarding Italy."
    )
    ]

agent_executor = create_conversational_retrieval_agent(llm, tools, memory_key='chat_history', verbose=True)

In [ ]:
memory

In [ ]:
agent_executor({"input": "what can I visit in India in 3 days?"})

In [ ]:
agent_executor({"input": "what is the current wheather in Delhi ?"})

In [ ]:
agent_executor({"input": "I'm travelling to Italy, can you give me some suggestions of the main attractions?"})